# Mind Map 3D Visualization

This notebook renders a **Zylvex Mind Mapper** mind map as an interactive 3D network graph.

**What you'll see:**
- Nodes positioned in 3D space using NetworkX's spring layout algorithm
- Node color reflects the BCI **focus score** (green → yellow → red)
- Node size reflects the **number of children** (more children = larger sphere)
- Hover tooltips show: node title, focus score, and depth level
- An interactive **Sandbox** section at the end lets you filter and re-render live

All data is hardcoded — no external APIs or files required.

In [ ]:
!pip install networkx plotly numpy ipywidgets --quiet

In [ ]:
import json
import warnings
import numpy as np
import networkx as nx
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

warnings.filterwarnings('ignore')

In [ ]:
# Sample mind map: 20 nodes in a hierarchical tree
SAMPLE_NODES = [
    {"id": "n0",  "title": "Machine Learning",       "parent_id": None,  "focus_score": 0.92, "depth": 0},
    {"id": "n1",  "title": "Neural Networks",         "parent_id": "n0",  "focus_score": 0.85, "depth": 1},
    {"id": "n2",  "title": "Supervised Learning",     "parent_id": "n0",  "focus_score": 0.78, "depth": 1},
    {"id": "n3",  "title": "Unsupervised Learning",   "parent_id": "n0",  "focus_score": 0.65, "depth": 1},
    {"id": "n4",  "title": "Reinforcement Learning",  "parent_id": "n0",  "focus_score": 0.71, "depth": 1},
    {"id": "n5",  "title": "Backpropagation",         "parent_id": "n1",  "focus_score": 0.88, "depth": 2},
    {"id": "n6",  "title": "Transformers",            "parent_id": "n1",  "focus_score": 0.94, "depth": 2},
    {"id": "n7",  "title": "CNNs",                    "parent_id": "n1",  "focus_score": 0.82, "depth": 2},
    {"id": "n8",  "title": "Linear Regression",       "parent_id": "n2",  "focus_score": 0.55, "depth": 2},
    {"id": "n9",  "title": "Decision Trees",          "parent_id": "n2",  "focus_score": 0.60, "depth": 2},
    {"id": "n10", "title": "K-Means Clustering",      "parent_id": "n3",  "focus_score": 0.48, "depth": 2},
    {"id": "n11", "title": "Autoencoders",            "parent_id": "n3",  "focus_score": 0.72, "depth": 2},
    {"id": "n12", "title": "Q-Learning",              "parent_id": "n4",  "focus_score": 0.66, "depth": 2},
    {"id": "n13", "title": "Policy Gradients",        "parent_id": "n4",  "focus_score": 0.58, "depth": 2},
    {"id": "n14", "title": "Attention Mechanism",     "parent_id": "n6",  "focus_score": 0.91, "depth": 3},
    {"id": "n15", "title": "BERT",                    "parent_id": "n6",  "focus_score": 0.89, "depth": 3},
    {"id": "n16", "title": "GPT Architecture",        "parent_id": "n6",  "focus_score": 0.95, "depth": 3},
    {"id": "n17", "title": "Gradient Descent",        "parent_id": "n5",  "focus_score": 0.80, "depth": 3},
    {"id": "n18", "title": "Dropout Regularization",  "parent_id": "n7",  "focus_score": 0.74, "depth": 3},
    {"id": "n19", "title": "Random Forests",          "parent_id": "n9",  "focus_score": 0.62, "depth": 3},
]

print(f'Loaded {len(SAMPLE_NODES)} nodes')
print(json.dumps(SAMPLE_NODES[:3], indent=2))

In [ ]:
# Build NetworkX directed graph
G = nx.DiGraph()

node_by_id = {n['id']: n for n in SAMPLE_NODES}

for node in SAMPLE_NODES:
    G.add_node(node['id'], **node)

for node in SAMPLE_NODES:
    if node['parent_id']:
        G.add_edge(node['parent_id'], node['id'])

# Count children for each node
children_count = {n: len(list(G.successors(n))) for n in G.nodes()}

print(f'Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}')
print('Children counts:', {k: v for k, v in children_count.items() if v > 0})

In [ ]:
# 3D spring layout using NetworkX
pos_3d = nx.spring_layout(G, dim=3, seed=42, k=1.5)

# Verify positions
sample_id = list(pos_3d.keys())[0]
print(f'Position of {sample_id}: {pos_3d[sample_id]}')
print(f'All nodes positioned: {len(pos_3d) == len(SAMPLE_NODES)}')

In [ ]:
def make_3d_figure(nodes, graph, positions, min_focus=0.0, min_size=4):
    """Build the Plotly 3D figure for the mind map."""
    filtered = [n for n in nodes if n['focus_score'] >= min_focus]
    filtered_ids = {n['id'] for n in filtered}

    children_cnt = {n: len([s for s in graph.successors(n) if s in filtered_ids]) for n in filtered_ids}

    # Edge traces (one trace with None separators)
    edge_x, edge_y, edge_z = [], [], []
    for src, tgt in graph.edges():
        if src in filtered_ids and tgt in filtered_ids:
            x0, y0, z0 = positions[src]
            x1, y1, z1 = positions[tgt]
            edge_x += [x0, x1, None]
            edge_y += [y0, y1, None]
            edge_z += [z0, z1, None]

    edge_trace = go.Scatter3d(
        x=edge_x, y=edge_y, z=edge_z,
        mode='lines',
        line=dict(color='rgba(99,102,241,0.4)', width=2),
        hoverinfo='none',
        name='Connections',
    )

    # Node traces
    node_x, node_y, node_z, node_colors, node_sizes, node_text = [], [], [], [], [], []
    for n in filtered:
        x, y, z = positions[n['id']]
        node_x.append(x)
        node_y.append(y)
        node_z.append(z)
        node_colors.append(n['focus_score'])
        size = max(min_size, min_size + children_cnt.get(n['id'], 0) * 3)
        node_sizes.append(size)
        node_text.append(
            f"<b>{n['title']}</b><br>"
            f"Focus: {n['focus_score']:.0%}<br>"
            f"Depth: {n['depth']}<br>"
            f"Children: {children_cnt.get(n['id'], 0)}"
        )

    node_trace = go.Scatter3d(
        x=node_x, y=node_y, z=node_z,
        mode='markers+text',
        marker=dict(
            size=node_sizes,
            color=node_colors,
            colorscale=[
                [0.0, '#ef4444'],   # low focus → red
                [0.5, '#f59e0b'],   # mid focus → yellow
                [1.0, '#10b981'],   # high focus → green
            ],
            cmin=0, cmax=1,
            colorbar=dict(title='Focus Score', x=1.02),
            opacity=0.92,
            line=dict(color='rgba(255,255,255,0.3)', width=1),
        ),
        text=[n['title'] for n in filtered],
        textposition='top center',
        textfont=dict(size=9, color='rgba(255,255,255,0.8)'),
        hovertext=node_text,
        hoverinfo='text',
        name='Nodes',
    )

    layout = go.Layout(
        title=dict(text=f'Mind Map 3D Visualization ({len(filtered)} nodes)', font=dict(color='#f8fafc', size=18)),
        paper_bgcolor='#0a0a0f',
        plot_bgcolor='#0a0a0f',
        scene=dict(
            xaxis=dict(showgrid=False, zeroline=False, showticklabels=False, title=''),
            yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, title=''),
            zaxis=dict(showgrid=False, zeroline=False, showticklabels=False, title=''),
            bgcolor='#0a0a0f',
        ),
        margin=dict(l=0, r=0, b=0, t=50),
        showlegend=False,
    )

    return go.Figure(data=[edge_trace, node_trace], layout=layout)


fig = make_3d_figure(SAMPLE_NODES, G, pos_3d)
fig.show()

In [ ]:
# Export to standalone HTML
fig.write_html('mind_map_3d.html', include_plotlyjs='cdn')
print('Exported to mind_map_3d.html')

## 🎛️ Sandbox — Interactive Controls

Use the sliders below to filter nodes by focus score and adjust the minimum node size, then re-render the 3D chart live.

In [ ]:
output = widgets.Output()

focus_slider = widgets.FloatSlider(
    value=0.0, min=0.0, max=1.0, step=0.05,
    description='Min Focus:', style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px'),
)
size_slider = widgets.IntSlider(
    value=4, min=4, max=20, step=1,
    description='Min Node Size:', style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px'),
)

def update(min_focus, min_size):
    with output:
        output.clear_output(wait=True)
        visible = [n for n in SAMPLE_NODES if n['focus_score'] >= min_focus]
        print(f'Visible nodes: {len(visible)} / {len(SAMPLE_NODES)}')
        f = make_3d_figure(SAMPLE_NODES, G, pos_3d, min_focus=min_focus, min_size=min_size)
        f.show()

widgets.interact(update, min_focus=focus_slider, min_size=size_slider)
display(output)